# 1. First steps

Clustrix runs an ordinary Python function somewhere else: in worker processes
on this machine, on a machine you reach over SSH, as SLURM jobs, or as a
Hugging Face Job. You mark the function with `@cluster` and call it as you
always did.

This first tutorial needs no cluster. It covers

- installing the package,
- decorating a function and calling it,
- finding out where it ran,
- `configure()`, which decides where the next call runs,
- `clustrix.explain()`, which says what clustrix will do with a function.

Every output below was produced by running this notebook.

## Install

```bash
pip install clustrix            # the library
pip install "clustrix[widget]"  # plus the Jupyter configuration widget
```

Python 3.10 or newer is required. If you intend to use a cluster, note now
that the Python on the cluster must be the same minor version as the one you
run here (3.12 with 3.12, 3.13 with 3.13); tutorial 3 explains why.

In [1]:
import sys
import clustrix

print("clustrix", clustrix.__version__, "on Python", sys.version.split()[0])

clustrix 0.2.0 on Python 3.13.18


## A function, decorated

Here is a small piece of numerical work, written with no thought for where it
will run.

In [2]:
import math


def mean_root(n):
    total = 0.0
    for i in range(n):
        total += math.sqrt(i)
    return total / n


mean_root(1_000_000)

666.6661664588418

Adding `@cluster` changes nothing about how the function is called or what it
returns.

In [3]:
from clustrix import cluster


@cluster
def mean_root(n):
    total = 0.0
    for i in range(n):
        total += math.sqrt(i)
    return total / n


mean_root(1_000_000)

666.6661664588418

## Where did it run?

So far nothing has told clustrix about a cluster, so the call ran on this
machine. A function can report where it is: the host name and the process ID
are enough to tell this notebook's process, another process on this machine,
and a remote machine apart.

In [4]:
import os
import socket


@cluster
def where_am_i():
    import os
    import socket

    return socket.gethostname(), os.getpid()


print("this notebook:", (socket.gethostname(), os.getpid()))
print("the function: ", where_am_i())

this notebook: ('vm', 14194)
the function:  ('vm', 14194)


Same host, same process. With no cluster configured, a decorated function
that has no loop to divide is simply called.

The decision is made from the configuration, which you can read at any time.
The field that matters is `cluster_host`: while it is empty, calls run here.

In [5]:
from clustrix import get_config

config = get_config()
print("cluster_type:", config.cluster_type)
print("cluster_host:", config.cluster_host)

cluster_type: slurm
cluster_host: None


`cluster_type` already says `slurm`. That is only the default value of the
field; without a `cluster_host` there is nothing to submit to and calls run
locally. To make the configuration say what is happening, set the type to
`local`.

## `configure()`

`configure()` changes settings for the rest of the session. Every keyword is
a field of `clustrix.ClusterConfig`. The four values `cluster_type` accepts
are listed in the package.

In [6]:
from clustrix import configure
from clustrix.config import SUPPORTED_CLUSTER_TYPES

print(SUPPORTED_CLUSTER_TYPES)

configure(cluster_type="local")
print(get_config().cluster_type)

('local', 'ssh', 'slurm', 'huggingface')
local


A type that is not in that list is refused when you set it, not when the first
job is submitted.

In [7]:
for name in ("local", "pbs"):
    try:
        configure(cluster_type=name)
        print(name, "- accepted")
    except ValueError as error:
        print(name, "-", error)

local - accepted
pbs - cluster_type='pbs' is not implemented. Clustrix ships no backend for it, because none has been verified against real hardware of that kind, so there is no code path that would run your function there. Support for it is tracked in issue #140. Supported types are: local, ssh, slurm, huggingface.


The decorator's own keywords describe the resources one job needs.

| keyword | meaning |
|---|---|
| `cores` | CPUs for a job; on this machine, the number of worker processes a loop is divided among |
| `memory` | memory for a job, e.g. `"8GB"` (used by SLURM) |
| `time` | time limit, e.g. `"01:00:00"` (used by SLURM) |
| `partition` | SLURM partition |
| `gpus` | GPUs for a job (tutorial 2 says how far this has been tested) |

Anything you leave out comes from the configuration: `default_cores`,
`default_memory`, `default_time`, `default_partition`.

In [8]:
print(config.default_cores, config.default_memory, config.default_time, config.default_partition)

4 8GB 01:00:00 None


Where the function runs is *not* a decorator keyword. Writing

```python
@cluster(cores=8, cluster_type="slurm")     # wrong: cluster_type is ignored here
def answer():
    return 42
```

does not send the function to SLURM. It runs wherever the configuration says,
and every call logs a warning:

```text
@cluster received unrecognised option(s) cluster_type; they have no effect.
Recognised extras: data, hf_flavor, hf_timeout, no_data
```

The backend is always chosen with `configure(cluster_type=...)`, so that the
same decorated function can be sent to a different place without editing it.

## The first loop divided among processes

`cores` matters on this machine too. If the function has a loop whose
iterations do not depend on one another, clustrix divides the loop among
`cores` worker processes. This function records which process ran each
iteration.

In [9]:
@cluster(cores=4)
def squares(numbers):
    import os

    results = []
    for number in numbers:
        results.append((number, number * number, os.getpid()))
    return results


for number, square, pid in squares([1, 2, 3, 4, 5, 6, 7, 8]):
    print(f"{number}^2 = {square:<3} computed in process {pid}")

print("this notebook is process", os.getpid())

1^2 = 1   computed in process 14214
2^2 = 4   computed in process 14214
3^2 = 9   computed in process 14212
4^2 = 16  computed in process 14212
5^2 = 25  computed in process 14215
6^2 = 36  computed in process 14215
7^2 = 49  computed in process 14213
8^2 = 64  computed in process 14213
this notebook is process 14194


The results are in the order the loop would have produced them, and the work
was done in other processes. Nothing about the function was written for
parallel execution. Tutorial 2 is about this feature: which loops are
divided, which are not, and how to tell.

## `clustrix.explain()`

`explain` reads a function and reports what clustrix would do with its loop,
without running it. It works on the decorated function or the plain one.

In [10]:
print(clustrix.explain(squares))

squares:
  line 6: `for number in numbers` -- SPLIT
      collected in order: results
      read-only (checked at run time): os
      private to each iteration: number
  code before the loop runs in the caller
  code after the loop runs in the caller


Reading the report:

- **SPLIT** — the loop on that line will be divided.
- **collected in order** — the names the loop writes results into. Their
  contents are put together in iteration order, so the result equals that of
  an ordinary run.
- **read-only (checked at run time)** — values every iteration reads. They
  are checked for changes while the loop runs.
- **private to each iteration** — names each iteration assigns for itself.

For a loop that cannot be divided, the report gives the line and the reason.

In [11]:
def compound(deposits):
    balance = 100.0
    history = []
    for deposit in deposits:
        balance = balance * 1.01 + deposit
        history.append(balance)
    return history


print(clustrix.explain(compound))

compound:
  line 4: `for deposit in deposits` -- not split
      line 5: balance is read before the iteration assigns it, so it carries a value from the previous iteration


Each iteration needs the balance the previous one left, so the iterations
cannot run separately. Decorating `compound` is still correct: it runs as one
piece. If you also asked for several cores, clustrix tells you once that they
were not used.

In [12]:
cluster(cores=4)(compound)([10.0, 10.0, 10.0])

@cluster(cores=4) has no effect here: compound has no loop that can be split. Locally, cores is the number of worker processes a loop in the function is divided among; with no loop that can be divided, the function runs once, in this process. clustrix.explain(function) says why a loop was not divided.


[111.0, 122.11, 133.3311]

## Seeing what clustrix is doing

Clustrix reports through Python's `logging` module under the logger name
`clustrix`. Warnings are shown by default. Turning on `INFO` shows the routine
decisions as well, which is useful while you are learning what it does.

In [13]:
import logging

logging.basicConfig(level=logging.WARNING, format="%(message)s")
logging.getLogger("clustrix").setLevel(logging.INFO)

cluster(cores=4)(compound)([10.0, 10.0, 10.0])

@cluster(cores=4) has no effect here: compound has no loop that can be split. Locally, cores is the number of worker processes a loop in the function is divided among; with no loop that can be divided, the function runs once, in this process. clustrix.explain(function) says why a loop was not divided.


compound runs unsplit.
compound:
  line 4: `for deposit in deposits` -- not split
      line 5: balance is read before the iteration assigns it, so it carries a value from the previous iteration


[111.0, 122.11, 133.3311]

In [14]:
logging.getLogger("clustrix").setLevel(logging.WARNING)

## Keeping a configuration

`configure()` lasts for the session. To keep settings, save them to a file.
Clustrix has a configuration directory, `~/.clustrix` unless the environment
variable `CLUSTRIX_CONFIG_DIR` names another, and a `config.yml` there is
loaded automatically whenever clustrix is imported.

In [15]:
from clustrix.config import get_config_dir, load_config, save_config

configure(default_cores=2, default_memory="4GB")
settings_file = get_config_dir() / "config.yml"
save_config(str(settings_file))

lines = settings_file.read_text().splitlines()
print(settings_file, "-", len(lines), "settings; the ones set above:")
print("\n".join(line for line in lines if line.startswith(("cluster_type", "default_cores", "default_memory"))))

/home/researcher/.clustrix/config.yml - 59 settings; the ones set above:
cluster_type: local
default_cores: 2
default_memory: 4GB


The directory is created if it does not exist, readable by you only.
Passwords and tokens are left out of the file unless you pass
`include_secrets=True`. Any other file can be read explicitly with
`load_config(path)`, which replaces the current settings with the file's.

In [16]:
configure(default_cores=4, default_memory="8GB")   # change the session...
load_config(str(settings_file))                    # ...and read the file back
print(get_config().default_cores, get_config().default_memory)

2 4GB


The file saved above would be loaded by every later session, including the
other tutorials, which assume the defaults. It was only an example, so remove
it again.

In [17]:
settings_file.unlink()
configure(default_cores=4, default_memory="8GB")
print(settings_file.exists())

False


One caution: a file named `clustrix.yml` in the *working directory* is also
picked up at import, with a warning, because a file that merely happens to be
where you started Python is not something you chose. Keep your settings in the
configuration directory.

In a notebook, the configuration widget (tutorial 6) keeps several named
configurations, called profiles, in the same directory.

## If something looks wrong

| You see | It means |
|---|---|
| `@cluster received unrecognised option(s) cluster_type` | The backend was given to the decorator. Use `configure(cluster_type=...)`. |
| `@cluster(cores=4) has no effect here: ... has no loop that can be split` | The function ran once, in this process. `clustrix.explain(function)` says why its loop was not divided. |
| `ValueError: ... cluster_type` naming `pbs`, `sge`, `kubernetes` or a cloud provider | Those backends are not part of clustrix. The supported values are in `clustrix.config.SUPPORTED_CLUSTER_TYPES`. |
| The function runs here although `cluster_type` is `ssh` or `slurm` | `cluster_host` is empty. Without a host there is nowhere to send the job. |

## Next

- [Tutorial 2](02_parallel_loops.ipynb): loops divided among workers, in detail.
- [Tutorial 3](03_ssh_cluster.ipynb): the same function on a remote machine.